In [ ]:
from pathlib import Path
import math
import pickle
import warnings

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

# Select datasets and metrics to display.
DATASETS = ['heart', 'diabetes_h', 'diabetes_130', 'obesity', 'obesity_bin', 'breast_cancer']
METRICS = {
    'accuracy': 'Accuracy',
    'f1': 'F1 score',
    'roc_auc': 'ROC-AUC',
}

# Run from the project root or from src/.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'results_metrics').is_dir() and (PROJECT_ROOT.parent / 'results_metrics').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULTS_DIR = PROJECT_ROOT / 'results_metrics' / 'current'

# Expected sizes keep missing intermediate results visible as gaps.
DATASET_SIZES = {
    'heart': [100, 500, 1000, 5000, 10000],
    'diabetes_h': [100, 500, 1000, 5000, 10000],
    'diabetes_130': [100, 500, 1000, 5000, 10000],
    'obesity': [100, 500, 1000, 1688],
    'obesity_bin': [100, 500, 1000, 1688],
    'breast_cancer': [100, 455],
}
DATASET_LABELS = {
    'heart': 'Heart', 'diabetes_h': 'Diabetes-H',
    'diabetes_130': 'Diabetes-130', 'obesity': 'Obesity',
    'obesity_bin': 'Obesity-Bin', 'breast_cancer': 'Breast Cancer',
}
MODEL_STYLES = {
    'lr':       {'label': 'LR',           'color': '#0072B2', 'marker': 'o'},
    'rf':       {'label': 'RF',           'color': '#D55E00', 'marker': 's'},
    'mlp':      {'label': 'MLP',          'color': '#009E73', 'marker': '^'},
    'nam':      {'label': 'NAM',          'color': '#CC79A7', 'marker': 'D'},
    'kan':      {'label': 'Logistic-KAN', 'color': '#E69F00', 'marker': 'v'},
    'kan_gam':  {'label': 'KAAM',         'color': '#332288', 'marker': 'P'},
    'ebm':      {'label': 'EBM',          'color': '#882255', 'marker': 'X'},
    'xgboost':  {'label': 'XGBoost',      'color': '#44AA99', 'marker': '<'},
    'lightgbm': {'label': 'LightGBM',     'color': '#999933', 'marker': '>'},
}
MODELS = list(MODEL_STYLES)  # Replace with a subset if desired.

# Paper-sized figures (inches and points).
FIGURE_WIDTH = 7.0
PANEL_HEIGHT = 2.7
MAX_COLUMNS = 2
FONT_SIZE = 9
TITLE_SIZE = 10
LEGEND_SIZE = 8
LINE_WIDTH = 1.3
MARKER_SIZE = 4
BAND_ALPHA = 0.16
X_SCALE = 'log'
SAVE_FIGURES = False
OUTPUT_DIR = PROJECT_ROOT / 'results_metrics' / 'figures'
OUTPUT_FORMAT = 'pdf'


# Sample-size analysis

The horizontal axis is the size of the selected training and validation set. The held-out test set is excluded. Each shaded band represents a saved 95% bootstrap interval; no model is trained and no metric is recalculated here.

Choose the datasets, metrics and visual styles in the first cell. Set `METRICS` to include `error_rate` if the desired quantity is `1 - accuracy`. Only load trusted pickle files.


In [ ]:
if not METRICS:
    raise ValueError('Select at least one metric in METRICS.')
if not RESULTS_DIR.is_dir():
    raise FileNotFoundError(f'Results directory not found: {RESULTS_DIR}')

unknown_datasets = set(DATASETS) - DATASET_SIZES.keys()
unknown_models = set(MODELS) - MODEL_STYLES.keys()
if unknown_datasets or unknown_models:
    raise ValueError(f'Unknown datasets: {unknown_datasets}; unknown models: {unknown_models}')

rows = []
missing = []
for dataset in DATASETS:
    for size in DATASET_SIZES[dataset]:
        for model in MODELS:
            path = RESULTS_DIR / str(size) / dataset / f'{model}.pkl'
            if not path.is_file():
                missing.append(f'{dataset}, N={size}, {model}: file not found')
                continue
            with path.open('rb') as file:
                result = pickle.load(file)
            means = result.get('metric_mean', {})
            intervals = result.get('metric_ci', {})
            is_95_percent = result.get('bootstrap', {}).get('ci') == 95
            for metric in METRICS:
                stored_metric = 'accuracy' if metric == 'error_rate' else metric
                if stored_metric not in means:
                    missing.append(f'{dataset}, N={size}, {model}: {metric} not found')
                    continue
                value = float(means[stored_metric])
                lower, upper = np.nan, np.nan
                if is_95_percent and stored_metric in intervals:
                    lower, upper = map(float, intervals[stored_metric])
                else:
                    missing.append(f'{dataset}, N={size}, {model}: 95% CI for {metric} not found')
                if metric == 'error_rate':
                    value, lower, upper = 1 - value, 1 - upper, 1 - lower
                rows.append({
                    'dataset': dataset, 'size': int(size), 'model': model,
                    'metric': metric, 'mean': value, 'lower': lower, 'upper': upper,
                })

results = pd.DataFrame(rows, columns=['dataset', 'size', 'model', 'metric', 'mean', 'lower', 'upper'])
if missing:
    print(f'Missing results or intervals ({len(missing)}):')
    for message in missing:
        print(f'  - {message}')
if results.empty:
    print('No matching metrics found. Check the configuration and results directory.')
else:
    print(f'Loaded {len(results)} metric values from {RESULTS_DIR}')


## Figures

Each model uses the same color and marker across panels and datasets. Lines and confidence bands are interrupted at missing sizes. A point without a saved 95% interval is still shown, without a band at that size.


In [ ]:
plot_style = {
    'font.size': FONT_SIZE,
    'axes.labelsize': FONT_SIZE,
    'axes.titlesize': TITLE_SIZE,
    'xtick.labelsize': FONT_SIZE,
    'ytick.labelsize': FONT_SIZE,
    'legend.fontsize': LEGEND_SIZE,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
}

with plt.rc_context(plot_style):
    for dataset in DATASETS:
        dataset_results = results[results['dataset'] == dataset]
        if dataset_results.empty:
            print(f'Skipping {dataset}: no selected results.')
            continue

        sizes = sorted(DATASET_SIZES[dataset])
        n_metrics = len(METRICS)
        n_cols = min(MAX_COLUMNS, n_metrics)
        n_rows = math.ceil(n_metrics / n_cols)
        fig, axes = plt.subplots(
            n_rows, n_cols,
            figsize=(FIGURE_WIDTH, PANEL_HEIGHT * n_rows),
            squeeze=False,
            layout='constrained',
        )
        axes = axes.ravel()
        plotted_models = set()

        for ax, (metric, metric_label) in zip(axes, METRICS.items()):
            for model in MODELS:
                points = dataset_results[
                    (dataset_results['metric'] == metric)
                    & (dataset_results['model'] == model)
                ].set_index('size').reindex(sizes)
                if points['mean'].notna().sum() == 0:
                    continue
                style = MODEL_STYLES[model]
                x = np.asarray(sizes)
                mean = points['mean'].to_numpy(dtype=float)
                lower = points['lower'].to_numpy(dtype=float)
                upper = points['upper'].to_numpy(dtype=float)
                ax.plot(
                    x, mean, color=style['color'], marker=style['marker'],
                    linewidth=LINE_WIDTH, markersize=MARKER_SIZE,
                )
                ax.fill_between(x, lower, upper, color=style['color'], alpha=BAND_ALPHA, linewidth=0)
                plotted_models.add(model)

            ax.set_title(metric_label)
            ax.set_xlabel('Training size (train + validation)')
            ax.set_ylabel(metric_label)
            ax.set_xscale(X_SCALE)
            ax.set_xticks(sizes, [f'{size:,}' for size in sizes])
            ax.tick_params(axis='x', which='minor', bottom=False)
            ax.grid(axis='y', alpha=0.25, linewidth=0.5)

        for ax in axes[n_metrics:]:
            ax.set_visible(False)

        handles = [
            Line2D(
                [0], [0], color=MODEL_STYLES[model]['color'],
                marker=MODEL_STYLES[model]['marker'], linewidth=LINE_WIDTH,
                markersize=MARKER_SIZE, label=MODEL_STYLES[model]['label'],
            )
            for model in MODELS if model in plotted_models
        ]
        if handles:
            fig.legend(handles=handles, loc='outside lower center', ncol=min(3, len(handles)), frameon=False)
        fig.suptitle(DATASET_LABELS.get(dataset, dataset), fontsize=TITLE_SIZE)

        if SAVE_FIGURES:
            OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
            output_path = OUTPUT_DIR / f'{dataset}_sample_sizes.{OUTPUT_FORMAT}'
            fig.savefig(output_path, bbox_inches='tight')
            print(f'Saved {output_path}')
        plt.show()
